# 階段 3：Context 預算實驗（步驟 1、2）

對應文件：`docs/03-stage-harness-1.md`。這份 notebook 做「量測」的部分：一份上千筆的 Trivy 報告直接塞進 context 會怎樣、裁剪與壓縮各省多少。狀態檢查點、權限、hooks 是應用程式碼，照文件寫在 `src/vuln_agent/`。

**前置**：階段 2 的 agent loop 能跑；本機裝好 [Trivy](https://trivy.dev/)。

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent  # kernel 的工作目錄是 notebooks/，往上一層才是專案根目錄

## 步驟 1：準備報告、量出基準

在終端機掃**自己的** image（參數以 Trivy 官方文件為準）：

```bash
mkdir -p data/reports/trivy
trivy image --format json --output data/reports/trivy/myapp.json <你的 image>
```

想要多一點結果，可以用舊版 base image 另 build 一個練習用 image。

### 1.1 統計報告內容

Trivy JSON 的弱點在 `Results[].Vulnerabilities[]`，常用欄位：`VulnerabilityID`、`PkgName`、`InstalledVersion`、`FixedVersion`、`Severity`（以你實際拿到的檔案為準）。

In [ ]:
REPORT = ROOT / "data" / "reports" / "trivy" / "myapp.json"
raw = REPORT.read_text()
report = json.loads(raw)

vulns = [v for r in report.get("Results", []) for v in (r.get("Vulnerabilities") or [])]
print("總筆數:", len(vulns))
print("各 severity:", Counter(v.get("Severity") for v in vulns))
print("不重複 CVE:", len({v.get("VulnerabilityID") for v in vulns}))
print(f"有 FixedVersion 的比例: {sum(bool(v.get('FixedVersion')) for v in vulns) / max(len(vulns), 1):.0%}")
print("檔案大小:", f"{len(raw) / 1024:.0f} KB")

### 1.2 整份報告有多少 tokens？

用 `count_tokens` 計算，不會生成回答，也不收費，只受每分鐘請求數限制。太大可能直接報錯，報錯本身也是一個數據點，記下來。

token 數由模型的 tokenizer 決定，tokenizer 會隨模型版本更新，各家模型都一樣。例如 2026-10 實測，Opus 5.5、Sonnet 5.5、Haiku 5.5 算這份報告的數字相同，上一代的 Sonnet 4.6 少約 15%；JSON、程式碼和中文的差距也不一樣。所以 token 預算、花費估算、context 上限的判斷，都要用實際呼叫的模型來算，換模型就重量，例如階段 8 換成開源小模型時。

In [ ]:
from vuln_agent.llm import client, DEFAULT_MODEL

try:
    n = client.messages.count_tokens(model=DEFAULT_MODEL, messages=[{"role": "user", "content": raw}])
    print("整份報告:", n.input_tokens, "tokens")
except Exception as e:
    print("超過上限或失敗:", e)

### 1.3 準備 5 題報告問題與標準答案

標準答案用程式算，不要用眼睛數；後面的天真版基準和三種策略比較都拿 `GOLD` 來跑。KEV 那題用 `src/vuln_agent/db.py` 的 `connect("agent")` 查 `kev` 表，資料庫要先起來、跑過階段 2 的 ETL。

In [ ]:
from vuln_agent.db import connect

# KEV 題要查階段 2 ETL 抓進資料庫的 kev 表
cve_ids = sorted({v["VulnerabilityID"] for v in vulns})
with connect("agent") as conn:
    in_kev = [r[0] for r in conn.execute("SELECT cve_id FROM kev WHERE cve_id = ANY(%s) ORDER BY cve_id", (cve_ids,))]

# 來源套件只記在 Results[].Packages[]，例如 Alpine 的 libssl3、libcrypto3 都來自 openssl
src = {p["Name"]: p.get("SrcName") for r in report.get("Results", []) for p in (r.get("Packages") or [])}

pkg_counts = Counter(v.get("PkgName") for v in vulns)
top = max(pkg_counts.values(), default=0)

GOLD = [
    {"question": "有幾個 Critical？",
     "answer": sum(v.get("Severity") == "CRITICAL" for v in vulns)},
    {"question": "openssl 相關的有哪些、修到哪一版？",
     "answer": sorted({(v["VulnerabilityID"], v.get("FixedVersion")) for v in vulns
                       if "openssl" in (src.get(v["PkgName"]) or v["PkgName"])})},
    {"question": "哪些在 KEV 裡、要先修？",
     "answer": in_kev},
    {"question": "哪個套件的弱點最多、有幾個？",
     "answer": sorted((p, n) for p, n in pkg_counts.items() if n == top)},  # 同票時全部列出
    {"question": "High 以上的弱點有幾個還沒有修補版本？",
     "answer": sum(v.get("Severity") in ("CRITICAL", "HIGH") and not v.get("FixedVersion") for v in vulns)},
]
for g in GOLD:
    print(g["question"], g["answer"])

這 5 題用來比較步驟 2 的三種 context 策略（天真版、裁剪版、裁剪加壓縮版）。每題考一種「要看到完整資料才答得對」的能力，策略漏看資料時答案就會出錯：

| 題目 | 考的能力 | 觀察重點 |
|---|---|---|
| 有幾個 Critical？ | 對整份報告計數 | 裁剪版只看到前 N 筆；模型有沒有注意到 `total` 和截斷說明，改用 severity 篩選再數 |
| openssl 相關的有哪些、修到哪一版？ | 依套件篩選、取欄位 | 套件名稱不一定有 openssl，Alpine 的 `libssl3`、`libcrypto3` 都來自 openssl，要看 `Packages[]` 的 `SrcName`；符合條件的列被截掉時，模型會不會用 package 篩選找回來 |
| 哪些在 KEV 裡、要先修？ | 報告與資料庫交叉比對 | 報告本身沒有 KEV 資訊，模型要呼叫階段 2 查 KEV 的工具；對照標準答案，看它有沒有憑記憶說出不在 KEV 的 CVE |
| 哪個套件的弱點最多、有幾個？ | 分組計數 | 要看過全部的列才能比較；天真版塞得進 context 就答得出來，裁剪版只看到一部分，容易答錯 |
| High 以上的弱點有幾個還沒有修補版本？ | 多條件篩選 | 要同時用 severity 和 FixedVersion 兩個條件 |

1.4 先用天真版記下基準，2.3 三種策略各跑一次後，逐題對照答對與否、input tokens、花費：裁剪省下的 token 有沒有換來答錯，哪一題最先錯，錯的原因是資料被截掉、模型沒去篩選，還是憑記憶回答。

答案依你掃的 image 而定，某題是 0 或空清單也是正常的標準答案；只是這種題目模型答「沒有」就會對，比較策略時鑑別力低，想看出差異可以用舊版 base image 另 build 一個 image 來掃。

### 1.4 天真版基準

天真版的工具是 `src/vuln_agent/report.py` 的 `read_scan_report()`：直接回傳整份 JSON，報告多大，context 就多大。路徑由同檔的 `report_path()` 檢查，只接受 `data/reports/` 底下的檔案。

`run_case` 用 `src/vuln_agent/agent.py` 的 `ask()` 跑一題，回傳答案、停止原因、步數、input tokens、花費、秒數。三種策略的差別都寫在 `STRATEGIES`：用哪組工具，以及傳給 `run_agent()` 的 `trim_tokens`（單一工具結果的上限）和 `context_tokens`（整段對話的上限）。問題前面附上報告路徑，模型才知道要讀哪份。每跑完一題，`run_case` 把結果追加寫進 `data/results/03_context_budget.jsonl`，2.3 用 `load_results()` 讀回三種策略的結果做表；付費跑出的結果存在檔案裡，kernel 重啟後不用重跑。結果附上 `stop`（停止原因）與 `steps`（呼叫模型幾次），回答是空的時，先看 `stop` 判斷原因：例如 `stop` 是 `max_tokens`、回應裡只有 thinking 和 tool_use，代表輸出上限在模型寫工具參數的途中就用完了。thinking、回答和工具參數共用 `src/vuln_agent/agent.py` 的 `MAX_TOKENS`，天真版要把報告裡幾十個 CVE 當參數傳給工具，輸出量比其他策略大。

第二格會呼叫付費 API。天真版每題至少把整份報告送兩次（讀報告一次、根據報告回答一次），先拿 1.2 量到的 tokens 和 `src/vuln_agent/llm.py` 的 `PRICES` 估一下花費再跑。答對與否對照 1.3 印出的 `GOLD` 判斷。

In [ ]:
from vuln_agent.agent import ask
from vuln_agent.report import NAIVE_REPORT_TOOLS, REPORT_TOOLS
from vuln_agent.tools import TOOLS

REPORT_PATH = REPORT.relative_to(ROOT).as_posix()  # data/reports/trivy/myapp.json
RESULTS_FILE = ROOT / "data" / "results" / "03_context_budget.jsonl"
STRATEGIES = {
    "naive": {"tools": {**TOOLS, **NAIVE_REPORT_TOOLS}},
    "trim": {"tools": {**TOOLS, **REPORT_TOOLS}, "trim_tokens": 2000},
    "trim+compact": {"tools": {**TOOLS, **REPORT_TOOLS}, "trim_tokens": 2000, "context_tokens": 5000},
}

def run_case(question: str, strategy: str) -> dict:
    """跑一題，結果追加寫進 RESULTS_FILE 再回傳。"""
    row = {"strategy": strategy, "question": question}
    try:
        r = ask(f"掃描報告：{REPORT_PATH}\n{question}", timeout=300, **STRATEGIES[strategy])
        row |= {"answer": r["answer"], "cost": r["cost_usd"],
                **{k: r[k] for k in ("stop", "steps", "trimmed", "compacted", "input_tokens", "output_tokens", "seconds")}}
    except Exception as e:  # 超過 context 上限這類錯誤也是數據點，記下來繼續跑下一題
        row["answer"] = f"錯誤：{e}"
    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with RESULTS_FILE.open("a") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return row

def load_results() -> list[dict]:
    """讀回 RESULTS_FILE；同一個策略的同一題跑過多次時，只留最後一次。"""
    latest = {}
    if RESULTS_FILE.exists():
        for line in RESULTS_FILE.read_text().splitlines():
            r = json.loads(line)
            latest[r["strategy"], r["question"]] = r
    return list(latest.values())


In [ ]:
# 會呼叫付費 API
for g in GOLD:
    r = run_case(g["question"], "naive")
    print(f"{g['question']}\n  回答：{r['answer'][:300]}\n  標準答案：{g['answer']}\n"
          f"  stop：{r.get('stop')}，steps：{r.get('steps')}，input tokens：{r.get('input_tokens')}，花費：{r.get('cost')}\n")


## 步驟 2：裁剪、摘要、token 預算

裁剪版不再把整份報告交給模型：`src/vuln_agent/report.py` 的 `import_report()` 把報告攤平存進 `findings` 表，只回傳 `report_id` 和統計摘要；模型再用 `query_findings()` 依 severity、套件、是否在 KEV 篩選與分頁，每頁附上 `total` 和 `next_cursor`，知道還有多少筆沒看。分組計數這類統計，模型可以用 `run_sql` 直接查 `findings` 表。

### 2.1 通用的工具結果裁剪

讀 `src/vuln_agent/context.py` 的 `trim_result()`：工具結果超過預算時，從最長的清單尾端截掉，並寫明截了多少，模型才知道要用篩選條件或分頁縮小範圍；沒有清單、或第一筆就放不下（例如天真版的整份報告）時，退回截斷 JSON 字串。呼叫 `run_agent()` 時給了 `trim_tokens`（單一工具結果的 token 上限），在工具結果接進 messages 之前就裁好。

token 用同檔的 `estimate_tokens()` 依字元數粗估。`CHARS_PER_TOKEN` 是用 `count_tokens` 實測出來的：JSON 約 1.8 字元一個 token、中文文件約 1.3，比常聽到的「英文約 4 字元一個 token」密很多。這個比例跟 1.2 說的一樣會隨 tokenizer 改變，換模型要重量。要精準就改呼叫 `count_tokens`，它不收費，但每次多一個 API 請求，也有每分鐘請求數上限。

下面用 `parse_trivy()` 攤平的報告實際裁一次，不呼叫 LLM。

In [ ]:
from vuln_agent.context import estimate_tokens, trim_result
from vuln_agent.report import parse_trivy

rows = parse_trivy(REPORT)
full = {"total": len(rows), "findings": rows}
trimmed = trim_result(full, max_tokens=2000)
print("裁剪前約", estimate_tokens(full), "tokens，裁剪後約", estimate_tokens(trimmed), "tokens")
print(trimmed.get("note", "沒有超過預算，原樣回傳"))

# 自我檢查：超過預算才裁，裁完連 note 一起算都不超過預算，total 保留原本的筆數
if "note" in trimmed:
    assert estimate_tokens(trimmed) <= 2000 and trimmed["total"] == len(rows)

### 2.2 對話層級的壓縮

讀 `src/vuln_agent/context.py` 的 `compact()`。`run_agent()` 收到 `context_tokens` 時，每輪呼叫模型前估算整段對話，超過就把整段換成一則摘要訊息，從這則摘要重新開始。摘要包含原始問題、模型已經說過的結論、每次工具呼叫的一行紀錄（工具、參數、結果開頭與長度），以及最近一次的工具結果。

壓縮是整段換掉，而不是只把較早的 `tool_result` 換成摘要、其他訊息照留，原因在 thinking block：Sonnet 5.5 這類模型回應裡的 thinking block，會綁定產生它時的對話前綴（system、tools 與之前每一則訊息）。改寫較早的訊息，之後的 thinking block 就全部失效；2026-08-31 以後建立的帳號會直接收到 400，較舊的帳號則是推理內容被丟掉。整段換成摘要、不回放舊訊息，就不會碰到這個檢查，代價是摘要以前的推理沒有帶過去，所以摘要要寫清楚哪些內容不能丟。裁剪也是同樣的道理，要在結果第一次接進 messages 之前做，歷史只追加、不改寫。

`compact()` 用程式產生摘要，不另外呼叫 LLM，結果固定也不花錢，缺點是摘要比較機械。API 也有內建的做法：context editing 清掉舊的工具結果，server-side compaction 由 API 產生摘要，兩者都不算改寫歷史。讀一下官方文件「Context editing」「Compaction」，想想它們和 `compact()` 各適合什麼情況。

下面用假的對話示範一次，不呼叫 LLM。

In [ ]:
from types import SimpleNamespace
from vuln_agent.context import compact

def tool_round(i: int, content: str) -> list:
    """一輪工具呼叫：assistant 的文字與 tool_use（模仿 SDK 回應物件的屬性），加上 user 的 tool_result。"""
    return [{"role": "assistant", "content": [
                SimpleNamespace(type="text", text=f"第 {i} 輪：查 HIGH 的弱點"),
                SimpleNamespace(type="tool_use", id=f"t{i}", name="query_findings",
                                input={"report_id": 1, "severity": "HIGH", "cursor": i * 20})]},
            {"role": "user", "content": [{"type": "tool_result", "tool_use_id": f"t{i}", "content": content}]}]

messages = [{"role": "user", "content": "有幾個 High？"}]
for i in range(1, 4):
    messages += tool_round(i, json.dumps({"total": len(rows), "findings": rows[:20]}, ensure_ascii=False))
out = compact(messages, max_tokens=1000)
print(len(messages), "則 →", len(out), "則，約", estimate_tokens(messages), "→", estimate_tokens(out), "tokens\n")
print(out[0]["content"][:1000])

# 自我檢查：換成一則 user 訊息，開頭是原始問題，三輪工具呼叫都有紀錄
assert len(out) == 1 and out[0]["role"] == "user" and out[0]["content"].startswith("有幾個 High？")
assert all(f"第 {i} 輪" in out[0]["content"] for i in range(1, 4))

### 2.3 三種策略比較

裁剪版、裁剪加壓縮版各跑一次 5 題，第一格會呼叫付費 API，但每次只送摘要與分頁結果，比天真版便宜很多。第二格不花錢，用 `load_results()` 讀回 1.4 的天真版和這兩種策略的結果，做成一張表。逐題對照 `GOLD`，看 1.3 表格列的觀察重點。除了答對與否，也比對天真版回答裡有、裁剪版沒有的資訊：`parse_trivy()` 只留 7 個欄位，各家的 CVSS、描述這類欄位攤平時就丟掉了，裁剪版的模型看不到，5 題的標準答案也考不到。攤平時保留哪些欄位，是在 token 與資訊量之間取捨。

表裡的 `trimmed` 是有幾個工具結果真的被 `trim_result()` 裁過，`compacted` 是 `compact()` 壓縮了幾次，兩者都是 0 時，省下的 token 來自攤平與篩選查詢，跟裁剪、壓縮無關。`trim_tokens`、`context_tokens` 的值在 1.4 的 `STRATEGIES`。`context_tokens` 設得比較小，5 題裡才會真的觸發壓縮；可以再調小，看哪一題最先答錯。

In [ ]:
# 會呼叫付費 API
for strategy in ("trim", "trim+compact"):
    for g in GOLD:
        run_case(g["question"], strategy)


In [ ]:
def show(rows: list[dict], cols: list[str]):
    print(" | ".join(cols))
    print(" | ".join("---" for _ in cols))
    for r in rows:
        print(" | ".join(str(r.get(c, "")) for c in cols))

results = load_results()
show(results, ["strategy", "question", "stop", "steps", "trimmed", "compacted", "input_tokens", "output_tokens", "cost", "seconds"])
print()
for r in results:
    print(f"[{r['strategy']}] {r['question']} → {r['answer'][:200]}")


**延伸實驗**：刻意把同一個 session 拉長到 20 輪（連續追問報告細節），每 5 輪問一題 `GOLD` 裡的題目，觀察答案品質從第幾輪開始下降、壓縮版有沒有撐比較久。

做完回到 `docs/03-stage-harness-1.md` 的步驟 3（狀態與檢查點）。